In [12]:
import requests
import pandas as pd
import us
from collections import Counter
from flair.data import Sentence
import pickle
from flair.nn import Classifier
HEADERS = {
    "User-Agent": "MyResearchBot/1.0 (ravim@vt.edu) python-requests",
    "Accept": "application/sparql-results+json",
}
SPARQL = "https://query.wikidata.org/sparql"
tagger = Classifier.load('ner-fast')
import sys
import numpy as np
from tqdm import tqdm
import time
import random
tqdm.pandas()
sys.modules["numpy._core"] = np.core
sys.modules["numpy._core.numeric"] = np.core.numeric
from pathlib import Path

###
# Get access token
session = requests.Session()
session.headers.update(HEADERS)

# Step 1: get login token
token_r = session.get("https://www.wikidata.org/w/api.php", params={
    "action": "query",
    "meta": "tokens",
    "type": "login",
    "format": "json"
})
token = token_r.json()["query"]["tokens"]["logintoken"]

# Step 2: login with token
session.post("https://www.wikidata.org/w/api.php", data={
    "action": "login",
    "lgname": "Meenuravi18@wildfire",
    "lgpassword": "vt9kqml328t409qqk2hh3lgs1v9o8rcm",
    "lgtoken": token,
    "format": "json"
})

2026-04-28 05:20:39,528 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>


/var/folders/4h/cm0jxy_x6vb7l398dj_2t2_00000gn/T/ipykernel_74791/1661187274.py:21: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric.
  sys.modules["numpy._core.numeric"] = np.core.numeric


<Response [200]>

In [2]:
STATE_TO_REGION = {
    "Connecticut": "Northeast", "Maine": "Northeast", "Massachusetts": "Northeast",
    "New Hampshire": "Northeast", "Rhode Island": "Northeast", "Vermont": "Northeast",
    "New Jersey": "Northeast", "New York": "Northeast", "Pennsylvania": "Northeast",
    "Illinois": "Midwest", "Indiana": "Midwest", "Michigan": "Midwest",
    "Ohio": "Midwest", "Wisconsin": "Midwest", "Iowa": "Midwest",
    "Kansas": "Midwest", "Minnesota": "Midwest", "Missouri": "Midwest",
    "Nebraska": "Midwest", "North Dakota": "Midwest", "South Dakota": "Midwest",
    "Delaware": "South", "Florida": "South", "Georgia": "South",
    "Maryland": "South", "North Carolina": "South", "South Carolina": "South",
    "Virginia": "South", "Washington DC": "South", "West Virginia": "South",
    "Alabama": "South", "Kentucky": "South", "Mississippi": "South",
    "Tennessee": "South", "Arkansas": "South", "Louisiana": "South",
    "Oklahoma": "South", "Texas": "South",
    "Arizona": "West", "Colorado": "West", "Idaho": "West", "Montana": "West",
    "Nevada": "West", "New Mexico": "West", "Utah": "West", "Wyoming": "West",
    "Alaska": "West", "California": "West", "Hawaii": "West",
    "Oregon": "West", "Washington": "West",
}

def get_region(hierarchy):
    state_names = {s.name for s in us.STATES}
    for h in hierarchy:
        if h in state_names:
            return STATE_TO_REGION.get(h)
    return None

In [3]:
state_names = {s.name for s in us.STATES} | {"United States"}

In [5]:
PROP_LABELS = {
    "P403": "mouth of the watercourse",
    "P610": "highest point",
    "P706": "located in physical feature",
    "P206": "located next to body of water",
}
def sparql_get(query, timeout=60):  # increase from 30 to 60
    for attempt in range(3):  # reduce retries to 3
        try:
            r = session.get(SPARQL, params={"query": query, "format": "json"}, timeout=timeout)
            if r.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"Rate limited. Sleeping {wait}s...")
                time.sleep(wait)
                continue
            if not r.text:
                time.sleep(5 * (attempt + 1))
                continue
            return r.json()
        except Exception as e:
            print(f"SPARQL error: {e}")
            time.sleep(5 * (attempt + 1))  # exponential backoff
    return {}
def flair_loc(sentence):
    locs = []
    for entity in sentence.get_spans("ner"):
        label = entity.get_label("ner")
        if label.value in ("LOC", "GPE") and label.score >= 0.85:
            locs.append(entity.text)
    return list(set(locs))
def level(h, state_names=None):
    if state_names is None:
        state_names = {s.name for s in us.STATES}
    h_lower = h.lower()
    if "united states" in h_lower:
        return 6
    if h in STATE_TO_REGION.values():
        return 5
    if h in state_names:
        return 4
    if any(k in h_lower for k in ["national park", "national forest", "wilderness", "reserve"]):
        return 1
    if any(k in h_lower for k in ["county", "parish", "borough"]):
        return 2
    return 0

def sort_hierarchy(hierarchy):
    state_names = {s.name for s in us.STATES}
    counties = [h for h in hierarchy if any(k in h.lower() for k in ["county", "parish", "borough"])]
    if counties:
        county = counties[0]
        matched_state = None
        for h in hierarchy:
            if h in state_names:
                result = sparql_get(f'ASK {{ ?c rdfs:label "{county}"@en ; wdt:P131 ?s . ?s rdfs:label "{h}"@en . }}')
                try:
                    if result.get("boolean"):
                        matched_state = h
                        break
                except Exception:
                    pass
        if matched_state:
            hierarchy = [h for h in hierarchy if h not in state_names or h == matched_state]
    # insert region for any state in hierarchy
    for h in list(hierarchy):
        if h in state_names:
            region = STATE_TO_REGION.get(h)
            if region and region not in hierarchy:
                hierarchy.append(region)
    return sorted(hierarchy, key=lambda h: level(h, state_names))

def search_candidates(query, retries=6):
    url = "https://www.wikidata.org/w/api.php"

    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": "en",
        "search": query,
        "type": "item",
        "limit": 10,
    }

    for attempt in range(retries):
        r = session.get(url, params=params, timeout=30)

        if r.status_code == 429:
            wait = 60 * (attempt + 1)
            print(f"Rate limited. Sleeping {wait}s...")
            time.sleep(wait)
            continue

        if r.status_code != 200:
            print("Wikidata status:", r.status_code)
            print(r.text[:300])
            time.sleep(10)
            continue

        try:
            data = r.json()
            time.sleep(0.5)   
            return [c["id"] for c in data.get("search", [])]
        except Exception:
            print("Non-JSON response")
            print(r.text[:300])
            time.sleep(30)

    return []

def get_hierarchy_for_qid(qid):
    query = f"""
    SELECT DISTINCT ?parent ?parentLabel WHERE {{
      {{ wd:{qid} wdt:P131+ ?parent . }}
      UNION
      {{ wd:{qid} wdt:P17 ?parent . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    rows = sparql_get(query).get("results", {}).get("bindings", [])
    hierarchy = list(dict.fromkeys(
        row["parentLabel"]["value"] for row in rows if "parentLabel" in row
    ))
    label_r = session.get(
        "https://www.wikidata.org/w/api.php",
        params={"action": "wbgetentities", "format": "json",
                "ids": qid, "props": "labels", "languages": "en"}
    ).json()

    place_label = label_r["entities"][qid]["labels"].get("en", {}).get("value")
    if place_label and place_label not in hierarchy:
        hierarchy.insert(0, place_label)
    return sort_hierarchy(hierarchy)

def resolve_by_prominence(candidates, method="population"):
    values = " ".join(f"wd:{q}" for q in candidates)
    if method == "population":
        query = f"""
        SELECT ?item (COALESCE(?pop, 0) AS ?score) WHERE {{
          VALUES ?item {{ {values} }}
          OPTIONAL {{ ?item wdt:P1082 ?pop . }}
          ?item wdt:P17 wd:Q30 .
        }} ORDER BY DESC(?score) LIMIT 1
        """
    else:
        query = f"""
        SELECT ?item (COUNT(?sitelink) AS ?score) WHERE {{
          VALUES ?item {{ {values} }}
          ?sitelink schema:about ?item .
          ?item wdt:P17 wd:Q30 .
        }} GROUP BY ?item ORDER BY DESC(?score) LIMIT 1
        """
    rows = sparql_get(query).get("results", {}).get("bindings", [])

    return rows[0]["item"]["value"].split("/")[-1] if rows else None
def search_cdp(place, state):
    query = f"""
    SELECT ?item WHERE {{
      ?item rdfs:label "{place}"@en .
      ?item wdt:P131+ ?s .
      ?s rdfs:label "{state}"@en .
    }} LIMIT 5
    """
    rows = sparql_get(query).get("results", {}).get("bindings", [])
    return rows[0]["item"]["value"].split("/")[-1] if rows else None
def get_hierarchy(place: str, state: str = None, context_text: str = None, tagger=tagger) -> dict:
    def build_output(qid, hierarchy, disambiguation):
        p31_query = f"""
            SELECT DISTINCT ?type ?typeLabel ?isNatural ?isCensusDesignated WHERE {{
            wd:{qid} wdt:P31 ?type .

            BIND(EXISTS {{
                ?type wdt:P279* ?naturalClass .
                VALUES ?naturalClass {{
                wd:Q4022      # river
                wd:Q355304    # watercourse
                wd:Q23397     # lake
                wd:Q8502      # mountain
                wd:Q8072      # volcano
                wd:Q35666     # glacier
                wd:Q23442     # island
                wd:Q8514      # desert
                wd:Q4421      # forest
                wd:Q39816     # valley
                wd:Q23442     # island
                wd:Q190429    # body of water
                wd:Q271669    # landform
                }}
            }} AS ?isNatural)

            BIND(EXISTS {{
                ?type wdt:P279* wd:Q498162
            }} AS ?isCensusDesignated)

            SERVICE wikibase:label {{
                bd:serviceParam wikibase:language "en".
            }}
            }}
            """
        rows = sparql_get(p31_query).get("results", {}).get("bindings", [])

        types = [row["typeLabel"]["value"].lower() for row in rows]
        feature_query = f"""
            SELECT DISTINCT ?prop ?featureLabel WHERE {{
            VALUES ?prop {{
                wdt:P403   # mouth of the watercourse
                wdt:P610   # highest point / summit
                wdt:P706   # located in/on physical feature
                wdt:P206   # located in or next to body of water
            }}

            wd:{qid} ?prop ?feature .

            SERVICE wikibase:label {{
                bd:serviceParam wikibase:language "en".
            }}
            }}
            LIMIT 1
            """

        feature_rows = sparql_get(feature_query).get("results", {}).get("bindings", [])
      

        is_natural = any(row.get("isNatural", {}).get("value") == "true" for row in rows)
        scope_type = types[0].replace("in the united states", "").strip() if types else None

        is_census = any(row.get("isCensusDesignated", {}).get("value") == "true" for row in rows)
        is_reservation = scope_type and any(x in scope_type for x in ["reservation", "territory", "tribal"])

        if is_natural and feature_rows:
            prop = feature_rows[0].get("prop", {}).get("value", "").split("/")[-1]
            disambiguation_hint = PROP_LABELS.get(prop, disambiguation)
        elif is_census or is_reservation:
            disambiguation_hint = "entire coverage"
        else:
            disambiguation_hint = disambiguation
        return {
            "place": place,
            "qid": qid,
            "hierarchy": hierarchy,
            "geo_scope": hierarchy[0] if hierarchy else place,
            "scope_type": types[0].replace("in the united states","") if types else None,
            "is_natural": is_natural,
            "is_us": "United States" in hierarchy,
            "disambiguation":  disambiguation_hint,
        }

    # Tier 1: explicit state
    if state or place in state_names:
        candidates = search_candidates(f"{place}, {state}" if state else place)
        if candidates:
            for qid in candidates:
                hierarchy = get_hierarchy_for_qid(qid)
                if state and state in hierarchy:
                    return build_output(qid, hierarchy, "explicit state")
                elif not state:
                    return build_output(candidates[0], hierarchy, "explicit state")
            # Tier 1b: CDP fallback
        else:
            qid = search_cdp(place, state)
            if qid:
                return build_output(qid, get_hierarchy_for_qid(qid), "cdp")
        return {"place": place, "error": "Could not resolve"}

    # Tier 2: extract state from context via Flair
    if context_text and tagger:
        sentence = Sentence(context_text)
        tagger.predict(sentence)
        locs = flair_loc(sentence)
        states_found = [us.states.lookup(loc).name for loc in locs if us.states.lookup(loc)]
        if states_found:
            best_state = Counter(states_found).most_common(1)[0][0]
            candidates = search_candidates(f"{place}, {best_state}")
            if candidates:
                hierarchy = get_hierarchy_for_qid(candidates[0])
                if "United States" in hierarchy:
                    count = Counter(states_found)
                    if len(count) > 1:
                        dis = f"ambiguous context - resolved by population"
                    else:
                        dis = f"Using context - {best_state}"
                    return build_output(candidates[0], hierarchy, dis)

    # Tier 3 & 4: prominence-based
    candidates = search_candidates(place)
    if not candidates:
        return {"place": place, "error": "Not found"}

    for method in ("population", "sitelinks"):
        qid = resolve_by_prominence(candidates, method)
        if qid:
            hierarchy = get_hierarchy_for_qid(qid)
            if hierarchy:
                return build_output(qid, hierarchy, method)

    return {"place": place, "error": "Could not resolve"}

In [6]:
with open('nodes/news_final.pkl', 'rb') as f1, open('nodes/pdf_final.pkl', 'rb') as f2:
    news = pickle.load(f1)
    pdf = pickle.load(f2)


In [6]:
# ct=0
# for i in news['extracted_geographies'].to_list():
#     if len(i)==0:
#         ct+=1
# print(ct)
# #2350 were no locs
# all_places = [p for locs in news["extracted_geographies"] for p in locs]
# print(f"Total: {len(all_places)}, Unique: {len(set(all_places))}")
#Total: 23559, Unique: 5827

In [7]:
news['extracted_geographies']=news['extracted_geographies'].progress_apply(lambda x: ['United States'] if len(x)==0 else x)
# news.head(7)

100%|██████████| 7311/7311 [00:00<00:00, 1011197.25it/s]


## Examples

In [76]:
# Tier 1 — explicit state
print(get_hierarchy("East coast"))

# Tier 2 — context
print(get_hierarchy("Springfield", context_text="The wildfire spread across Illinois and Missouri counties..."))

# # Tier 3 — no hints, fully automatic
print(get_hierarchy("Springfield"))

# Regular city
print(get_hierarchy("Los Angeles"))

# County
print(get_hierarchy("Cook County", state="Illinois"))

{'place': 'East coast', 'qid': 'Q4268', 'hierarchy': ['East Coast of the United States', 'United States'], 'geo_scope': 'East Coast of the United States', 'scope_type': 'geographic location', 'is_natural': False, 'is_us': True, 'disambiguation': 'population'}
{'place': 'Springfield', 'qid': 'Q135615', 'hierarchy': ['Springfield', 'Christian County', 'Greene County', 'Missouri', 'Midwest', 'United States'], 'geo_scope': 'Springfield', 'scope_type': 'city ', 'is_natural': False, 'is_us': True, 'disambiguation': 'ambiguous context - resolved by population'}
{'place': 'Springfield', 'qid': 'Q135615', 'hierarchy': ['Springfield', 'Christian County', 'Greene County', 'Missouri', 'Midwest', 'United States'], 'geo_scope': 'Springfield', 'scope_type': 'city ', 'is_natural': False, 'is_us': True, 'disambiguation': 'population'}
{'place': 'Los Angeles', 'qid': 'Q104994', 'hierarchy': ['Los Angeles County', 'California', 'West', 'United States'], 'geo_scope': 'Los Angeles County', 'scope_type': 'c

In [77]:
print(get_hierarchy("Yakama Indian Reservation"))

{'place': 'Yakama Indian Reservation', 'qid': 'Q3457242', 'hierarchy': ['Yakama Indian Reservation', 'Yakima County', 'Klickitat County', 'Washington', 'West', 'United States'], 'geo_scope': 'Yakama Indian Reservation', 'scope_type': 'indian reservation of the united states', 'is_natural': False, 'is_us': True, 'disambiguation': 'entire coverage'}


In [69]:
print(get_hierarchy("Marina del Rey"))

{'place': 'Marina del Rey', 'qid': 'Q988140', 'hierarchy': ['Marina del Rey', 'Los Angeles County', 'California', 'West', 'United States'], 'geo_scope': 'Marina del Rey', 'scope_type': 'census-designated place ', 'is_natural': False, 'is_us': True, 'disambiguation': 'entire coverage'}


In [65]:
print(get_hierarchy("Smoky Mountain"))

{'place': 'Smoky Mountain', 'qid': 'Q1360486', 'hierarchy': ['Great Smoky Mountains', 'Tennessee', 'North Carolina', 'South', 'United States'], 'geo_scope': 'Great Smoky Mountains', 'scope_type': 'mountain range', 'is_natural': True, 'is_us': True, 'disambiguation': 'highest point'}


In [66]:
print(get_hierarchy("Charles River"))

{'place': 'Charles River', 'qid': 'Q794927', 'hierarchy': ['Charles River', 'Massachusetts', 'Northeast', 'United States'], 'geo_scope': 'Charles River', 'scope_type': 'river', 'is_natural': True, 'is_us': True, 'disambiguation': 'mouth of the watercourse'}


In [67]:
print(get_hierarchy("Mississippi River"))

{'place': 'Mississippi River', 'qid': 'Q1497', 'hierarchy': ['Mississippi River', 'United States'], 'geo_scope': 'Mississippi River', 'scope_type': 'river', 'is_natural': True, 'is_us': True, 'disambiguation': 'mouth of the watercourse'}


## Non-time consuming run

In [ ]:

tqdm.pandas()
import json
CACHE_FILE = Path("geo_cache.json")
if CACHE_FILE.exists():
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        cache = json.load(f)
else:
    cache = {}
geo = pd.read_csv("US/US.txt", sep="\t", header=None,
                  names=["geonameid","name","asciiname","alternatenames",
                         "lat","lon","feature_class","feature_code","country",
                         "cc2","admin1","admin2","admin3","admin4",
                         "population","elevation","dem","timezone","modified"])
def norm_place(place):
    return str(place).strip()
state_code_to_name = {s.abbr: s.name for s in us.STATES}

all_places = list(set(
    [norm_place(p) for locs in news["extracted_geographies"] for p in locs if norm_place(p)] +
    [norm_place(p) for locs in pdf["extracted_geographies"] for p in locs if norm_place(p)]
))
admin2 = pd.read_csv("US/admin2Codes.txt", sep="\t", header=None,
                     names=["code", "name", "ascii_name", "geonameid"])
admin2_lookup = dict(zip(admin2["code"], admin2["name"]))
FEATURE_CODE_HINTS = {
    "STM": "mouth of the watercourse",
    "RVR": "mouth of the watercourse",
    "LK": "body of water",
    "MT": "highest point",
    "PKS": "highest point",
    "PRK": "entire coverage",
    "RES": "entire coverage",
    "RESV": "entire coverage",
}
# rebuild cache with deduplication
for place in tqdm(all_places):
    matches = geo[geo["name"].str.lower() == place.lower()]
    if matches.empty:
        continue
    row = matches.sort_values("population", ascending=False).iloc[0]
    state = state_code_to_name.get(row["admin1"], row["admin1"])
    admin2_code = f"US.{row['admin1']}.{int(row['admin2'])}" if pd.notna(row["admin2"]) else None
    county = admin2_lookup.get(admin2_code)
    region = STATE_TO_REGION.get(state)
    hierarchy = sort_hierarchy(list(dict.fromkeys([h for h in [place, county, state, region, "United States"] if h and isinstance(h, str)])))
    cache[place] = {
        "place": place,
        "hierarchy": hierarchy,
        "geo_scope": hierarchy[0],
        "scope_type": row["feature_code"],
        "is_natural": row["feature_class"] == "T",
        "is_us": True,
        "disambiguation": FEATURE_CODE_HINTS.get(row["feature_code"], "population"),
    }

with open(CACHE_FILE, "w") as f:
    json.dump(cache, f, ensure_ascii=False, indent=2)
print(f"Cached: {len(cache)}/{len(all_places)}")

In [ ]:
def parse_regional_phrase(place):
    state_names = {s.name for s in us.STATES}
    for state in state_names:
        if state in place:
            region = STATE_TO_REGION.get(state)
            hierarchy = sort_hierarchy([h for h in [place, state, region, "United States"] if h and isinstance(h, str)])
            return {
                "place": place,
                "hierarchy": hierarchy,
                "geo_scope": place,
                "scope_type": "region",
                "is_natural": False,
                "is_us": True,
                "disambiguation": "regional phrase",
            }
    return None

missing = [p for p in all_places if p not in cache]
print(f"Still missing: {len(missing)}")

for place in tqdm(missing):
    result = parse_regional_phrase(place)
    if result:
        cache[place] = result

with open(CACHE_FILE, "w") as f:
    json.dump(cache, f, ensure_ascii=False, indent=2)
print(f"Cached after regional phrase: {len(cache)}/{len(all_places)}")

Still missing: 3666


100%|██████████| 3666/3666 [00:00<00:00, 140062.29it/s]

Cached after regional phrase: 3599/7265


In [ ]:
missing = [p for p in all_places if p not in cache]
print(missing)

['Scenic River', 'Las Conchas Fire', 'D-Santa Monica', 'Kahoma Village', 'Pinar', 'Millis Swamp Road', 'Porto Velho', 'Behchoko', 'Kenai-Kodiak', 'Degarra', 'Duplin', 'Webber Lake Basin', 'Normétal', 'Chiquitania', 'Grant Counties', 'C.W. Avery', 'Agwato Lake', 'Eastern Australia', 'Pacific West', 'Black Hills Coniferous Forest Province', 'R. Magney State Park', 'Central America', 'Banksia Bluff', 'Minn', 'La Niña', 'Interstate 70', 'Basin           Mountains', 'Grey', 'Goomburra', 'Apgar Village', 'Lesvos', 'Santa Anita Avenue', 'Tehachapis', 'Meadow Ridge Way SOUTH', 'Collett Ridge Fire', 'Graying', 'Kate                                              Missoula', 'Tiller Trail', 'Forrestfield', 'Guy Fawkes River', 'CONNOLLY', 'Highway 16', 'Shan', 'Tallac', 'Informaiton', 'FS Road 1911    Tire Mountain Trailhead    Vivian Lake Trailhead    Diamond Lake    Thielsen View    Broken Arrow Campgrounds    Castle Creek Trailhead    Skimmerhorn Trailhead    Fish Lake Trailhead    Beaver Swamp T

In [ ]:
KEEP = ["hierarchy", "geo_scope", "scope_type", "is_natural", "is_us", "disambiguation"]
def resolve_row(row):
    results = []
    for place in row["extracted_geographies"]:
        place = norm_place(place)
        result = cache.get(place, {})
        if result and "error" not in result:
            results.append({k: result[k] for k in KEEP})
    # update extracted_geographies to only keep resolved ones
    row["extracted_geographies"] = [p for p in row["extracted_geographies"] if norm_place(p) in cache and "error" not in cache.get(norm_place(p), {})]
    return results
news["geo_mappings"] = news.progress_apply(resolve_row, axis=1)

100%|██████████| 7311/7311 [00:00<00:00, 26528.93it/s]


In [ ]:
news

,source,title,url,published_date,folder,extracted_geographies,Text,embedding_bge,embedding_bge_base,geo_mappings
0,https://headwaterseconomics.org/natural-hazard...,New research shows where wildfire mitigation c...,https://headwaterseconomics.org/natural-hazard...,2023-01-05 17:40:50+00:00,natural-hazards,"[California, United States, Montana, Arizona, ...",The federal government allocates vast amounts ...,"[-0.025733936578035355, -0.00678191939368844, ...","[0.01520454976707697, 0.0038301339372992516, -...","[{'hierarchy': ['California', 'West', 'United ..."
1,https://headwaterseconomics.org/natural-hazard...,New research shows where wildfire mitigation c...,https://headwaterseconomics.org/natural-hazard...,2023-01-05 17:40:50+00:00,natural-hazards,"[Montana, Northern California, Idaho, West, Wa...",Department of Agriculture’s Office of Inspecto...,"[-0.024573158472776413, 0.017168717458844185, ...","[0.0065810419619083405, -0.01915942132472992, ...","[{'hierarchy': ['Montana', 'West', 'United Sta..."
2,https://headwaterseconomics.org/natural-hazard...,New research shows where wildfire mitigation c...,https://headwaterseconomics.org/natural-hazard...,2023-01-05 17:40:50+00:00,natural-hazards,[United States],2023. “The Economic Incidence of Wildfire Supp...,"[0.013801665045320988, -0.001129874144680798, ...","[0.029995029792189598, 0.0280434712767601, 0.0...","[{'hierarchy': ['00', 'United States'], 'geo_s..."
3,https://headwaterseconomics.org/natural-hazard...,New research shows where wildfire mitigation c...,https://headwaterseconomics.org/natural-hazard...,2023-01-05 17:40:50+00:00,natural-hazards,[Colorado],Limitations The analysis in this post is based...,"[-0.0524384006857872, 0.02183299884200096, 0.0...","[-0.005320822354406118, 0.019476886838674545, ...","[{'hierarchy': ['Colorado', 'West', 'United St..."
4,https://headwaterseconomics.org/natural-hazard...,New research shows where wildfire mitigation c...,https://headwaterseconomics.org/natural-hazard...,2023-01-05 17:40:50+00:00,natural-hazards,[United States],Acknowledgments\n\nIn addition to their valuab...,"[-0.02109595760703087, 0.09811549633741379, -0...","[-0.005499211605638266, 0.0018088180804625154,...","[{'hierarchy': ['00', 'United States'], 'geo_s..."
...,...,...,...,...,...,...,...,...,...,...
7306,https://www.npr.org/2025/01/31/nx-s1-5276397/w...,"Farmworkers feed the country, but who protects...",https://www.npr.org/2025/01/31/nx-s1-5276397/w...,2025-01-31 00:00:00,nx-s1-5276397,"[Oxnard, Calif, Los Angeles County, Ventura Co...","Farmworkers feed the country, but who protects...","[0.018416794016957283, -0.016738982871174812, ...","[0.07241406291723251, -0.028616666793823242, -...","[{'hierarchy': ['Oxnard', 'Ventura County', 'C..."
7307,https://www.npr.org/2025/01/31/nx-s1-5276397/w...,"Farmworkers feed the country, but who protects...",https://www.npr.org/2025/01/31/nx-s1-5276397/w...,2025-01-31 00:00:00,nx-s1-5276397,"[Camarillo, Northern L.A. County, Los Angeles,...",is to the state's agricultural region. Drive a...,"[-0.0026630042120814323, 0.001136625069193542,...","[0.06479275226593018, -0.008711450733244419, -...","[{'hierarchy': ['Camarillo', 'Ventura County',..."
7308,https://www.npr.org/2025/01/31/nx-s1-5276397/w...,"Farmworkers feed the country, but who protects...",https://www.npr.org/2025/01/31/nx-s1-5276397/w...,2025-01-31 00:00:00,nx-s1-5276397,[United States],"These are folks without a social safety net,"" ...","[-0.030867312103509903, 0.05298491567373276, -...","[0.05857665091753006, -0.03455138951539993, -0...","[{'hierarchy': ['00', 'United States'], 'geo_s..."
7309,https://www.npr.org/2025/01/31/nx-s1-5276397/w...,"Farmworkers feed the country, but who protects...",https://www.npr.org/2025/01/31/nx-s1-5276397/w...,2025-01-31 00:00:00,nx-s1-5276397,[California],The outlook for California wildfires is grim s...,"[-0.026421887800097466, -0.0020182535517960787...","[0.022825190797448158, -0.062497612088918686, ...","[{'hierarchy': ['Cali

In [ ]:
news[news['extracted_geographies'].isna()]

,source,title,url,published_date,folder,extracted_geographies,Text,embedding_bge,embedding_bge_base,geo_mappings


In [ ]:
for i, row in news.iterrows():
    for mapping in row["geo_mappings"]:
        missing_keys = [k for k in KEEP if k not in mapping]
        if missing_keys or not mapping.get("hierarchy") or not mapping.get("geo_scope"):
            print(f"Row {i}: {mapping}")
            break

In [ ]:
for i, row in news.tail(5).iterrows():
    print(f"Row {i}: {row['extracted_geographies']}")
    for mapping in row["geo_mappings"]:
        print(f"  {mapping['geo_scope']} -> {mapping['hierarchy']}")
    print("====")

Row 7306: ['Oxnard', 'Calif', 'Los Angeles County', 'Ventura County']
  Oxnard -> ['Oxnard', 'Ventura County', 'California', 'West', 'United States']
  Los Angeles County -> ['Los Angeles County', 'California', 'West', 'United States']
  Ventura County -> ['Ventura County', 'California', 'West', 'United States']
====
Row 7307: ['Camarillo', 'Northern L.A. County', 'Los Angeles', 'California', 'Oxnard', 'Oregon', 'L.A. County', 'Washington', 'Irvine', 'Northern Los Angeles County', 'L.A.', 'Ventura County']
  Camarillo -> ['Camarillo', 'Ventura County', 'California', 'West', 'United States']
  Los Angeles -> ['Los Angeles', 'California', 'West', 'United States']
  California -> ['California', 'West', 'United States']
  Oxnard -> ['Oxnard', 'Ventura County', 'California', 'West', 'United States']
  Oregon -> ['Oregon', 'West', 'United States']
  Washington -> ['Washington', 'West', 'United States']
  Irvine -> ['Irvine', 'California', 'West', 'United States']
  Ventura County -> ['Ventur

In [ ]:
news.to_pickle("nodes/news_final_with_hierarchies.pkl")

In [ ]:
def resolve_row_pdf(row):
    results = []
    folder_state = row["folder"] if row["folder"] != "United States" else None
    for place in row["extracted_geographies"]:
        place = norm_place(place)
        result = cache.get(place, {})
        if result and "error" not in result:
            result = dict(result)
            hierarchy = result.get("hierarchy", [])
            # only override if not a natural feature disambiguation
            if result.get("disambiguation") not in FEATURE_CODE_HINTS.values():
                if folder_state and folder_state in hierarchy:
                    result["disambiguation"] = f"explicit state - {folder_state}"
                elif not folder_state:
                    result["disambiguation"] = "Resolving to United States"
            results.append({k: result[k] for k in KEEP if k in result})
    return results

pdf["geo_mappings"] = pdf.progress_apply(resolve_row_pdf, axis=1)


100%|██████████| 5664/5664 [00:00<00:00, 34942.10it/s]


In [ ]:
pdf[pdf['extracted_geographies'].isna()]

,source,start_page,end_page,folder,extracted_geographies,Text,embedding_bge,embedding_bge_base,geo_mappings


In [ ]:
for i, row in pdf.iterrows():
    for mapping in row["geo_mappings"]:
        missing_keys = [k for k in KEEP if k not in mapping]
        if missing_keys or not mapping.get("hierarchy") or not mapping.get("geo_scope"):
            print(f"Row {i}: {mapping}")
            break

In [ ]:
for i, row in pdf.iterrows():
    print(f"Row {i}: {row['extracted_geographies']}")
    for mapping in row["geo_mappings"]:
        print(f"  {mapping['geo_scope']} -> {mapping['hierarchy']} [{mapping.get('disambiguation')}]")
    print("====")

Row 0: ['Missouri', 'Ozarks']
  Missouri -> ['Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
  Ozarks -> ['Ozarks', 'Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
====
Row 1: ['Missouri']
  Missouri -> ['Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
====
Row 2: ['Missouri']
  Missouri -> ['Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
====
Row 3: ['Missouri']
  Missouri -> ['Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
====
Row 4: ['Missouri']
  Missouri -> ['Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
====
Row 5: ['Missouri']
  Missouri -> ['Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
====
Row 6: ['Missouri']
  Missouri -> ['Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
====
Row 7: ['Missouri']
  Missouri -> ['Missouri', 'Midwest', 'United States'] [explicit state - Missouri]
====
Row 8: ['Missouri']
  Missouri -> [

In [ ]:
pdf

,source,start_page,end_page,folder,extracted_geographies,Text,embedding_bge,embedding_bge_base,geo_mappings
0,Missouri/MoFRAS_0.pdf,1,12,Missouri,"[Missouri, Ozarks]",2010\n Missouri’s Forest Res...,"[-0.03424540534615517, 0.06734155118465424, 0....","[-0.021990643814206123, -0.02395980805158615, ...","[{'hierarchy': ['Missouri', 'Midwest', 'United..."
1,Missouri/MoFRAS_0.pdf,1,12,Missouri,[Missouri],Maintenance of Forest Contribution to Global C...,"[-0.021710919216275215, 0.022526130080223083, ...","[-0.014079088345170021, -0.04596194252371788, ...","[{'hierarchy': ['Missouri', 'Midwest', 'United..."
2,Missouri/MoFRAS_0.pdf,1,12,Missouri,[Missouri],Improve air quality and conserve energy\n ...,"[-0.009289962239563465, 0.02078832872211933, -...","[-0.00010305528121534735, -0.03276599943637848...","[{'hierarchy': ['Missouri', 'Midwest', 'United..."
3,Missouri/MoFRAS_0.pdf,1,12,Missouri,[Missouri],Strategy:\n\n 1. Assessment findings were ...,"[-0.022280070930719376, 0.0258120596408844, -0...","[-0.012356702238321304, -0.01699560508131981, ...","[{'hierarchy': ['Missouri', 'Midwest', 'United..."
4,Missouri/MoFRAS_0.pdf,1,12,Missouri,[Missouri],There is no net loss of Missouri’s total fores...,"[0.07706274837255478, 0.05835429206490517, -0....","[-0.02811952494084835, -0.06032869219779968, 0...","[{'hierarchy': ['Missouri', 'Midwest', 'United..."
...,...,...,...,...,...,...,...,...,...
5659,New Mexico/wildfire_agtab_2018.pdf,1,8,New Mexico,[New Mexico],"Wildfire, 15. Out, 16. Feet, 17.","[-0.029990481212735176, 0.05361329764127731, 0...","[0.017439410090446472, -0.021826045587658882, ...","[{'hierarchy': ['New Mexico', 'West', 'United ..."
5660,New Mexico/wildfire_agtab_2018.pdf,1,8,New Mexico,[New Mexico],"Weather. 7-C, 8-B, 9-D, 10-D\n ...","[0.0012159852776676416, 0.030964303761720657, ...","[0.0023207077756524086, 0.011013385839760303, ...","[{'hierarchy': ['New Mexico', 'West', 'United ..."
5661,New Mexico/wildfire_agtab_2018.pdf,1,8,New Mexico,[New Mexico],"Lighters, 6.","[-0.01580268330872059, 0.007295439951121807, 0...","[0.01874169148504734, 0.04687980189919472, 0.0...","[{'hierarchy': ['New Mexico', 'West', 'United ..."
5662,New Mexico/wildfire_agtab_2018.pdf,1,8,New Mexico,[New Mexico],"Sleeping, 7. Quickly,\n D. Keeps ...","[-0.05345318093895912, 0.022705765441060066, 0...","[-0.0006878797430545092, 0.005619741044938564,...","[{'hierarchy': ['New Mexico', 'West', 'United ..."


In [ ]:
pdf.to_pickle("nodes/pdf_final_with_hierarchies.pkl")